# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The Random Forest model from Week 5 is used to rank content that may require review. The model is intended to support prioritization rather than make automatic decisions.

The ranked queue groups content into action levels based on the predicted likelihood of decline and supporting performance signals. The reason codes help explain why a page appears in the queue so that a reviewer can quickly understand the recommendation.

| Priority | Recommended Action | Example Reason Code |
|----------|--------------------|---------------------|
| High | Review immediately | High predicted decline risk, declining impressions, aging content |
| Medium | Refresh content | Moderate decline risk, lower CTR, stable but weakening performance |
| Low | Monitor | Low predicted decline risk, no immediate action required |

These recommendations are decision-support only. Final actions should be determined through human review using current content quality, business priorities, and editorial judgment.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier


df = pd.read_csv("content_refresh_anonymized.csv")


df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

groups = df["client_id"]


X = df.drop(columns=[
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
])

y = df["is_declining_label"]

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf.fit(X_train, y_train)


decline_probability = rf.predict_proba(X_test)[:, 1]

action_queue = X_test.copy()

action_queue["Decline Probability"] = decline_probability


action_queue = action_queue.sort_values(
    "Decline Probability",
    ascending=False
)


action_queue["Priority"] = pd.cut(
    action_queue["Decline Probability"],
    bins=[0, 0.5, 0.8, 1],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)


action_queue["Recommended Action"] = action_queue["Priority"].map({
    "High": "Review immediately",
    "Medium": "Refresh content",
    "Low": "Monitor"
})


action_queue["Reason Code"] = action_queue["Priority"].map({
    "High": "High predicted decline probability",
    "Medium": "Moderate predicted decline probability",
    "Low": "Low predicted decline probability"
})

action_queue = action_queue[
    [
        "Priority",
        "Decline Probability",
        "Recommended Action",
        "Reason Code"
    ]
]

action_queue.head(10)

,Priority,Decline Probability,Recommended Action,Reason Code
19110,High,0.99,Review immediately,High predicted decline probability
3329,High,0.99,Review immediately,High predicted decline probability
29681,High,0.98,Review immediately,High predicted decline probability
18822,High,0.98,Review immediately,High predicted decline probability
10171,High,0.98,Review immediately,High predicted decline probability
5410,High,0.98,Review immediately,High predicted decline probability
5353,High,0.97,Review immediately,High predicted decline probability
25063,High,0.97,Review immediately,High predicted decline probability
17707,High,0.97,Review immediately,High predicted decline probability
3211,High,0.97,Review immediately,High predicted decline probability


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


This action playbook is designed to support editorial decision-making by ranking content based on its predicted likelihood of decline. The recommendations help identify pages that may benefit from review, refresh, or continued monitoring.

The model provides decision support only. It does not determine whether content should be updated, removed, or published. Final decisions should be made by human reviewers using additional context such as business priorities, seasonal trends, content quality, and subject matter expertise.

The predictions are based on patterns observed in the available dataset and may not generalize to new content, different organizations, or future performance without additional validation.

In [10]:
action_queue.groupby("Priority", observed=False)["Decline Probability"].agg(["count", "mean"]).round(2)

,count,mean
Priority,,
Low,2702,0.30
Medium,2662,0.66
High,799,0.86


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


Before acting on any recommendation, a human reviewer should verify that the content is accurate, relevant, and aligned with current business priorities. Additional factors such as seasonal trends, recent updates, marketing campaigns, and subject matter expertise should also be considered.

The action playbook should not automatically update, remove, or publish content based solely on model predictions. The model identifies patterns associated with declining performance, but final editorial decisions require human judgment.

In [11]:
no_go_list = pd.DataFrame({
    "Human Review Required": [
        "Verify content accuracy",
        "Check business priorities",
        "Review seasonal trends",
        "Approve final action"
    ],
    "Never Automate": [
        "Content updates",
        "Content removal",
        "Publishing decisions",
        "Editorial approval"
    ]
})

no_go_list

,Human Review Required,Never Automate
0,Verify content accuracy,Content updates
1,Check business priorities,Content removal
2,Review seasonal trends,Publishing decisions
3,Approve final action,Editorial approval


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The action playbook should be monitored regularly to ensure that its recommendations remain reliable over time. Model performance should be reviewed using metrics such as accuracy, precision, recall, and F1 score, along with feedback from human reviewers.

If content characteristics, user behavior, or business goals change significantly, the model should be retrained using updated data and validated before deployment. Regular monitoring helps ensure that the playbook continues to provide useful decision support.

In [12]:
monitoring_plan = pd.DataFrame({
    "Activity": [
        "Review model performance",
        "Collect reviewer feedback",
        "Check data changes",
        "Retrain and validate"
    ],
    "Frequency": [
        "Regularly",
        "Regularly",
        "When new data is available",
        "After significant changes"
    ]
})

monitoring_plan

,Activity,Frequency
0,Review model performance,Regularly
1,Collect reviewer feedback,Regularly
2,Check data changes,When new data is available
3,Retrain and validate,After significant changes


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


The ranked action queue is exported as a CSV file so it can be reviewed by editors or integrated into downstream workflows. The exported file contains the priority level, predicted decline probability, recommended action, and reason code for each content item.

In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

action_queue.to_csv(
    "work/outputs/action_queue.csv",
    index=False
)

pd.read_csv("work/outputs/action_queue.csv").head()

,Priority,Decline Probability,Recommended Action,Reason Code
0,High,0.99,Review immediately,High predicted decline probability
1,High,0.99,Review immediately,High predicted decline probability
2,High,0.98,Review immediately,High predicted decline probability
3,High,0.98,Review immediately,High predicted decline probability
4,High,0.98,Review immediately,High predicted decline probability


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.